In [1]:
import asyncio
from nats.aio.client import Client as NATS
from nats.js.errors import NotFoundError

In [ ]:

STREAM = "TEST_STREAM"
SUBJECT = "test.subject"


async def setup_stream(js):
    try:
        await js.add_stream(
            name=STREAM,
            subjects=[SUBJECT],
            storage="file"
        )
        print("Stream created")
    except Exception:
        print("Stream already exists")


async def publisher(js):
    print("\n📤 Publishing messages...")
    for i in range(5):
        await js.publish(
            SUBJECT,
            f"message-{i}".encode()
        )
        print(f"Published message-{i}")


async def consumer(js):
    print("\n📥 Starting consumer...")

    sub = await js.pull_subscribe(
        SUBJECT,
        durable="test-consumer"
    )

    while True:
        msgs = await sub.fetch(1, timeout=2)

        for msg in msgs:
            data = msg.data.decode()
            print(f"Received: {data}")

            # 🔥 Simulate failure for one message
            if data == "message-2":
                print("Simulating failure → NOT ACKING")
                continue

            await msg.ack()
            print("ACK sent")  

# 

# await publisher(js)

# # Run consumer
# await consumer(js)


In [8]:
from nats.aio.client import Client as NATS

nc = NATS()
await nc.connect("nats://localhost:4222")

js = nc.jetstream()

In [9]:
await setup_stream(js)

Stream created


In [10]:
await publisher(js)


📤 Publishing messages...
Published message-0
Published message-1
Published message-2
Published message-3
Published message-4


In [11]:
sub = await js.pull_subscribe(
    "test.subject",          # your subject
    durable="test-consumer"  # existing durable
)

In [ ]:
msgs = await sub.fetch(1)
for msg in msgs:
    print("Received:", msg.data.decode())

    # ❌ DO NOT ACK
    await msg.ack()